In [ ]:
%pip install -q langchain-openai langchain-core requests

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

d:\My projects\LangChain\LangChain Models\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
    model= 'gpt-4o-mini',
    base_url="https://openrouter.ai/api/v1"
)

In [3]:
@tool

def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetchers the currency conversion factor between given base currency and a target currency
    """
    url = f'https://v6.exchangerate-api.com/v6/e1287784a2dd7c8856903ee8/pair/{base_currency}/{target_currency}'
    
    
    response = requests.get(url)
    
    return response.json()
    

In [4]:
result = get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'BDT'})

In [5]:
print(result)

{'result': 'success', 'documentation': 'https://www.exchangerate-api.com/docs', 'terms_of_use': 'https://www.exchangerate-api.com/terms', 'time_last_update_unix': 1780617601, 'time_last_update_utc': 'Fri, 05 Jun 2026 00:00:01 +0000', 'time_next_update_unix': 1780704001, 'time_next_update_utc': 'Sat, 06 Jun 2026 00:00:01 +0000', 'base_code': 'USD', 'target_code': 'BDT', 'conversion_rate': 122.6933}


In [7]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [8]:
@tool

def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    This function will convert the base currency to target currency based on the given conversion rate
    """
    
    return base_currency_value * conversion_rate


In [9]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [10]:
 get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'BDT'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1780617601,
 'time_last_update_utc': 'Fri, 05 Jun 2026 00:00:01 +0000',
 'time_next_update_unix': 1780704001,
 'time_next_update_utc': 'Sat, 06 Jun 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'BDT',
 'conversion_rate': 122.6933}

In [11]:
convert.invoke({'base_currency_value': 10, 'conversion_rate': 122.69})

1226.9

In [12]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [14]:
message  = [HumanMessage('What is the conversion factor between BDT and USD, and based on this convert 10 BDT to USD')]

In [15]:
message

[HumanMessage(content='What is the conversion factor between BDT and USD, and based on this convert 10 BDT to USD', additional_kwargs={}, response_metadata={})]

In [16]:
ai_message = llm_with_tools.invoke(message)

In [17]:
message.append(ai_message)

In [19]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'BDT', 'target_currency': 'USD'},
  'id': 'call_gpwKpeHgOnvKMBCy5ztoes26',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_HpYIjyF23uPGXUoVmU1HByPF',
  'type': 'tool_call'}]

In [18]:
message

[HumanMessage(content='What is the conversion factor between BDT and USD, and based on this convert 10 BDT to USD', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 115, 'total_tokens': 168, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 4.905e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 4.905e-05, 'upstream_inference_prompt_cost': 1.725e-05, 'upstream_inference_completions_cost': 3.18e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_7fa0fbf23d', 'id': 'gen-1780651097-98w0MuR14WXSNr7bGlbI', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e

In [20]:
import json

for tool_call in ai_message.tool_calls:
    
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        
        message.append(tool_message1)
        
    if tool_call['name'] == 'convert':
        
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        message.append(tool_message2)

In [23]:
message

[HumanMessage(content='What is the conversion factor between BDT and USD, and based on this convert 10 BDT to USD', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 115, 'total_tokens': 168, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 4.905e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 4.905e-05, 'upstream_inference_prompt_cost': 1.725e-05, 'upstream_inference_completions_cost': 3.18e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_7fa0fbf23d', 'id': 'gen-1780651097-98w0MuR14WXSNr7bGlbI', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e

In [25]:
result = llm_with_tools.invoke(message)

In [26]:
result.content

'The conversion factor between Bangladeshi Taka (BDT) and United States Dollar (USD) is approximately 0.00815. \n\nBased on this conversion factor, 10 BDT is equivalent to approximately 0.0815 USD.'